In [61]:
import sys 
sys.path.append("../")
import pandas as pd
import plotly.graph_objects as go
import datetime as dt
from technicals.indicators import RSI 
from technicals.patterns import apply_patterns
from plotting import CandlePlot

In [21]:
df_raw = pd.read_pickle("../data/GBP_JPY_M5.pkl")

In [22]:
df_raw.shape

(143, 14)

In [23]:
df_an = df_raw.copy()
df_an.reset_index(drop=True, inplace=True)

In [24]:
df_an = RSI(df_an)

In [25]:
df_an.tail()

,volume,time,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,RSI_14
138,268,2016-12-22 23:00:00+00:00,144.400,144.429,144.394,144.422,144.326,144.384,144.326,144.383,144.473,144.496,144.449,144.461,59.413382
139,193,2016-12-27 03:00:00+00:00,143.939,143.969,143.936,143.955,143.921,143.952,143.916,143.937,143.957,143.986,143.954,143.973,57.155430
140,97,2016-12-28 04:00:00+00:00,144.490,144.508,144.458,144.460,144.471,144.491,144.439,144.441,144.508,144.528,144.475,144.480,58.971271
141,249,2016-12-29 05:00:00+00:00,142.971,143.008,142.967,142.980,142.952,142.987,142.947,142.959,142.990,143.031,142.987,143.002,52.013742
142,248,2016-12-30 06:00:00+00:00,143.360,143.374,143.303,143.308,143.338,143.352,143.275,143.286,143.383,143.397,143.328,143.331,53.327962


In [26]:
df_an = apply_patterns(df_an)

In [27]:
df_an['EMA_200'] = df_an.mid_c.ewm(span=200, min_periods=200).mean()

In [28]:
df_an.columns

Index(['volume', 'time', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h',
       'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c', 'RSI_14',
       'body_lower', 'body_upper', 'body_bottom_perc', 'body_top_perc',
       'body_perc', 'direction', 'body_size', 'low_change', 'high_change',
       'body_size_change', 'mid_point', 'mid_point_prev_2', 'body_size_prev',
       'direction_prev', 'direction_prev_2', 'body_perc_prev',
       'body_perc_prev_2', 'HANGING_MAN', 'SHOOTING_STAR', 'SPINNING_TOP',
       'MARUBOZU', 'ENGULFING', 'TWEEZER_TOP', 'TWEEZER_BOTTOM',
       'MORNING_STAR', 'EVENING_STAR', 'EMA_200'],
      dtype='object')

In [29]:
our_cols = ['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'ask_c', 'bid_c', 'ENGULFING', 'direction', 'EMA_200', 'RSI_14']

In [30]:
df_slim = df_an[our_cols].copy()
df_slim.dropna(inplace=True)
df_slim.reset_index(drop=True, inplace=True)

In [31]:
df_slim.head()

,time,mid_o,mid_h,mid_l,mid_c,ask_c,bid_c,ENGULFING,direction,EMA_200,RSI_14


In [32]:
BUY = 1
SELL = -1
NONE = 0
RSI_LIMIT = 50.0 

def apply_signal(row):
    if row.ENGULFING == True:
        if row.direction == BUY and row.mid_l > row.EMA_200:
            if row.RSI_14 > RSI_LIMIT:
                return BUY
        if row.direction == SELL and row.mid_h < row.EMA_200:
            if row.RSI_14 < RSI_LIMIT:
                return SELL
    return NONE

In [33]:
df_slim["SIGNAL"] = df_slim.apply(apply_signal, axis=1)

In [34]:
df_slim["SIGNAL"].value_counts()

Series([], Name: count, dtype: int64)

In [35]:
LOSS_FACTOR = -1.0
PROFIT_FACTOR = 1.5 

def apply_take_profit(row):
    if row.SIGNAL != NONE:
        return(row.mid_c - row.mid_o) * 1.5 + row.mid_c
    else:
        return 0.0
    
def apply_stop_loss(row):
    if row.SIGNAL != NONE:
        return row.mid_o
    else:
        return 0.0

In [36]:
df_slim["TP"] = df_slim.apply(apply_take_profit, axis = 1)
df_slim["SL"] = df_slim.apply(apply_stop_loss, axis=1)

In [38]:
df_plot = df_slim
cp = CandlePlot(df_plot, candles=True)

trades = cp.df_plot[df_plot.SIGNAL != NONE]

markers = ['mid_c', 'TP', 'SL']
marker_colors = ['#0000FF', '#00FF00', '#FF0000']

for i in range(3):
    cp.fig.add_trace(go.Scatter(
        x = trades.sTime,
        y = trades[markers[i]],
        mode = 'markers',
        marker = dict(color=marker_colors[i], size=12)
    ))
cp.show_plot(line_traces=["EMA_200"], sec_traces=['RSI_14'])

In [39]:
class Trade:
    def __init__self(self, row):
        self.running = True
        self.start_index = row.name
        self.start_price = row.mid_c
        self.trigger_price = row.mid_c
        self.SIGNAL = row.SIGNAL
        self.TP = row.TP
        self.SL = row.SL
        self.result = 0.0
        self.end_ttime = row.time
        self.start_time = row.time
        self.duration = 0

    def close_trade(self, row, result, trigger_price):
        self.running = False
        self.result = result
        self.end_time = row.time
        self.trigger_price = trigger_price

    def update(self, row):
        self.duration += 1
        if self.SIGNAL == BUY:
            if row.mid_h >= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.mid_h)
            elif row.mid_l <= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.mid_l)
        elif self.SIGNAL == SELL:
            if row.mid_l <= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.mid_l)
            elif row.mid_h >= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.mid_h)

In [40]:
open_trades = [] 
closed_trades = []

for index, row in df_slim.iterrows():
    for ot in open_trades:
        ot.update(row)
        if ot.running == False:
            closed_trades.append(ot)
    open_trades = [x for x in open_trades if x.running == True]

    if row.SIGNAL != NONE:
        open_trades.append(Trade(row))

In [41]:
df_results = pd.DataFrame.from_dict([vars(x) for x in closed_trades])

In [43]:
df_results.sort_values(by="start_index", inplace=True)

KeyError: 'start_index'

In [44]:
df_m5 = pd.read_pickle("../data/EUR_USD_M5.pkl")

In [45]:
df_m5.shape

(143, 14)

In [46]:
from dateutil import parser

In [56]:
time_min = parser.parse("2016-12-15T10:00:00Z")
time_max = parser.parse("2016-12-15T11:00:00Z")
df_m5_s = df_m5[(df_m5.time >= time_min) & (df_m5.time <= time_max)]
df_raw_s = df_raw[(df_raw.time >= time_min) & (df_raw.time <= time_max)]


In [57]:
df_m5_s

,volume,time,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c


In [58]:
df_raw_s

,volume,time,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c


In [59]:
df_m5_slim = df_m5[['time', 'mid_h', 'mid_l']].copy()

In [60]:
df_m5_slim.head()

,time,mid_h,mid_l
0,2016-06-07 00:00:00+00:00,1.13664,1.13628
1,2016-06-08 01:00:00+00:00,1.13653,1.13620
2,2016-06-09 02:00:00+00:00,1.14129,1.14090
3,2016-06-10 03:00:00+00:00,1.12962,1.12950
4,2016-06-13 06:00:00+00:00,1.12599,1.12508


In [64]:
df_signals = df_slim[df_slim.SIGNAL != NONE].copy()

In [67]:
df_signals['m5_start'] = [x + dt.timedelta(hours=1) for x in df_signals.time]
df_signals['start_index_h1'] = df_signals.index
df_signals.head()

,time,mid_o,mid_h,mid_l,mid_c,ask_c,bid_c,ENGULFING,direction,EMA_200,RSI_14,SIGNAL,TP,SL,m5_start,start_index_h1


In [68]:
df_signals.drop(['time', 'mid_o', 'mid_h', 'mid_l', 'ask_c', 'bid_c', 'ENGULFING', 'EMA_200', 'RSI_14', 'direction'], axis=1, inplace=True)

In [69]:
df_signals.rename(columns={'mid_c':'start_price','m5_start':'time'})

,start_price,SIGNAL,TP,SL,time,start_index_h1


In [70]:
merged = pd.merge(left=df_m5_slim, right=df_signals, on='time', how='left')

KeyError: 'time'

In [71]:
merged.fillna(0, inplace=True)

NameError: name 'merged' is not defined

In [72]:
merged.SIGNAL = merged.SIGNAL.astype(int)
merged.start_index_h1 = merged.sIGNAL.astype(int)

NameError: name 'merged' is not defined

In [73]:
class TradeM5:
    def __init__self(self, row):
        self.running = True
        self.start_index_m5 = row.name
        self.start_index_h1 = row.start_index_h1
        self.start_price = row.start_price
        self.trigger_price = row.start_price
        self.SIGNAL = row.SIGNAL
        self.TP = row.TP
        self.SL = row.SL
        self.result = 0.0
        self.end_ttime = row.time
        self.start_time = row.time
        self.duration = 0

    def close_trade(self, row, result, trigger_price):
        self.running = False
        self.result = result
        self.end_time = row.time
        self.trigger_price = trigger_price

    def update(self, row):
        self.duration += 1
        if self.SIGNAL == BUY:
            if row.mid_h >= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.mid_h)
            elif row.mid_l <= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.mid_l)
        elif self.SIGNAL == SELL:
            if row.mid_l <= self.TP:
                self.close_trade(row, PROFIT_FACTOR, row.mid_l)
            elif row.mid_h >= self.SL:
                self.close_trade(row, LOSS_FACTOR, row.mid_h)

In [75]:
open_trades_m5 = [] 
closed_trades = []

for index, row in merged.iterrows():
    if row.SIGNAL != NONE:
        open_trades_m5.append(TradeM5(row))
        
    for ot in open_trades_m5:
        ot.update(row)
        if ot.running == False:
            closed_trades.append(ot)
    open_trades_m5 = [x for x in open_trades_m5 if x.running == True]

    

NameError: name 'merged' is not defined